# Compute imagery statistics

Summarize finite index values in the acquired composites and export
`outputs/composites/imagery_stats.csv`. Composites already use the acquisition
footprint; this notebook does not apply another polygon mask. See the
[Imagery Acquisition and Preparation white paper](../docs/white-papers/Imagery-Acquisition-and-Preparation.md)
for the acquisition and index methodology.

Running the export replaces the existing CSV.


In [1]:
import re
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import rasterio

from s2grab.config import AcquisitionConfig
from s2grab.footprint import FOOTPRINT_RULE

# Suppress internal numpy/rasterio shape deprecation warnings if present
warnings.filterwarnings("ignore", category=DeprecationWarning)

ROOT = Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "Run notebooks from the notebooks/ folder"
ACQUISITION = AcquisitionConfig.from_yaml(
    ROOT / "configs" / "imagery.yaml", project_root=ROOT
)
TARGET_INDICES = ACQUISITION.indices
EXPECTED_FOOTPRINT_RULE = FOOTPRINT_RULE

/home/tshihadah/code/wadi-gaza-research/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 · Configuration and Paths

In [2]:
# Input and output directory locations
COMPOSITES_DIR = ACQUISITION.output_dir
QA_MANIFEST_PATH = COMPOSITES_DIR / "qa_manifest.csv"
OUTPUT_CSV_PATH = COMPOSITES_DIR / "imagery_stats.csv"

# Filename pattern: <site>_<YYYY>-<MM>.tif
FILENAME_PATTERN = re.compile(r"^(?P<site>.+)_(?P<year>\d{4})-(?P<month>\d{2})\.tif$")

## 2 · Discovery and Site Footprint Reference

Acquisition (notebook 00) masks every composite to its site's pixel-center footprint and records the rule (`FOOTPRINT_RULE`) and footprint size (`FOOTPRINT_PIXELS`) as GeoTIFF tags. This step reads those tags, which supply the site-area denominator, and checks they agree across all dates for a site.


In [3]:
def discover_composite_tifs(composites_dir: Path) -> list[Path]:
    """Find site composite GeoTIFFs matching the expected naming pattern."""
    return sorted(
        path for path in composites_dir.glob("*/*.tif")
        if FILENAME_PATTERN.match(path.name)
    )


def read_site_footprints(tif_paths: list[Path]) -> tuple[dict[str, int], dict[str, float]]:
    """Read each site's footprint pixel count and pixel area from composite tags."""
    site_area_pixels: dict[str, int] = {}
    site_pixel_area: dict[str, float] = {}
    site_grids: dict[str, tuple[object, object, tuple[int, int]]] = {}

    for path in tif_paths:
        match = FILENAME_PATTERN.match(path.name)
        if match is None:
            continue
        site = match.group("site")
        with rasterio.open(path) as src:
            tags = src.tags()
            grid = (src.crs, src.transform, src.shape)
            pixel_area = float(abs(src.transform.a * src.transform.e - src.transform.b * src.transform.d))
        if tags.get("FOOTPRINT_RULE") != EXPECTED_FOOTPRINT_RULE or "FOOTPRINT_PIXELS" not in tags:
            raise ValueError(f"Composite is missing pixel-center footprint tags: {path}")
        pixels = int(tags["FOOTPRINT_PIXELS"])
        if site not in site_area_pixels:
            site_area_pixels[site] = pixels
            site_pixel_area[site] = pixel_area
            site_grids[site] = grid
        elif (pixels, grid) != (site_area_pixels[site], site_grids[site]):
            raise ValueError(f"Footprint or grid changed across dates for {site}: {path}")

    return site_area_pixels, site_pixel_area


tif_files = discover_composite_tifs(COMPOSITES_DIR)
site_area_pixels_by_site, site_pixel_area = read_site_footprints(tif_files)

print(f"Discovered {len(tif_files)} composite rasters across {len(site_area_pixels_by_site)} sites:")
for site, site_area_pixels in sorted(site_area_pixels_by_site.items()):
    total_area_ha = site_area_pixels * site_pixel_area[site] / 10_000.0
    total_area_km2 = site_area_pixels * site_pixel_area[site] / 1_000_000.0
    print(
        f"  {site:<22}: {site_area_pixels:>6} pixels "
        f"({total_area_ha:>8.2f} ha | {total_area_km2:>6.3f} km2)"
    )

Discovered 294 composite rasters across 7 sites:
  AlAhrashProtectorate  :  33848 pixels (  338.09 ha |  3.381 km2)
  BesorStream           :  11318 pixels (  113.03 ha |  1.130 km2)
  HaBesorNationalPark   :  91755 pixels (  918.64 ha |  9.186 km2)
  HolotNizzanim         : 119868 pixels ( 1198.48 ha | 11.985 km2)
  HolotPalmahim         :  62192 pixels (  620.89 ha |  6.209 km2)
  NahalRubin            :   7860 pixels (   78.56 ha |  0.786 km2)
  WadiGaza              :  28413 pixels (  284.33 ha |  2.843 km2)


## 3 · Extract Statistics Across Sites, Dates, and Indices

In [4]:
def extract_imagery_statistics(
    tif_paths: list[Path],
    site_area_pixels_by_site: dict[str, int],
    site_pixel_area: dict[str, float],
    qa_manifest_path: Path,
    indices: tuple[str, ...] = TARGET_INDICES,
) -> pd.DataFrame:
    """Summarize each index over the finite (footprint-masked) composite pixels."""
    if not qa_manifest_path.exists():
        raise FileNotFoundError(f"Missing composite QA manifest: {qa_manifest_path}")

    qa_df = pd.read_csv(qa_manifest_path, dtype={"site": str, "window": str})
    required_qa_columns = {"site", "window", "status", "n_scenes", "mean_cloud"}
    missing_qa_columns = required_qa_columns.difference(qa_df.columns)
    if missing_qa_columns:
        raise ValueError(f"QA manifest is missing columns: {sorted(missing_qa_columns)}")
    qa_df["window"] = qa_df["window"].str.strip()
    if qa_df.duplicated(["site", "window"]).any():
        raise ValueError("QA manifest contains duplicate site/window rows")
    qa_lookup = qa_df.set_index(["site", "window"])

    records: list[dict[str, Any]] = []

    for path in tif_paths:
        match = FILENAME_PATTERN.match(path.name)
        if match is None:
            continue

        site = match.group("site")
        year = int(match.group("year"))
        month = match.group("month")
        window = f"{year}-{month}"
        if site not in site_area_pixels_by_site:
            raise ValueError(f"No footprint pixel count for {site!r}")
        if (site, window) not in qa_lookup.index:
            raise ValueError(f"No QA manifest row for {site}, {window}")

        qa_info = qa_lookup.loc[(site, window)]
        if qa_info["status"] != "ok":
            raise ValueError(f"QA status is not ok for {site}, {window}: {qa_info['status']}")

        pixel_area_m2 = site_pixel_area[site]
        site_area_pixels = site_area_pixels_by_site[site]
        total_area_ha = site_area_pixels * pixel_area_m2 / 10_000.0
        total_area_km2 = site_area_pixels * pixel_area_m2 / 1_000_000.0

        with rasterio.open(path) as src:
            descriptions = list(src.descriptions)
            missing_indices = [name for name in indices if name not in descriptions]
            if missing_indices:
                raise ValueError(f"Missing index bands {missing_indices} in {path}")
            band_numbers = [descriptions.index(name) + 1 for name in indices]
            data = src.read(band_numbers, masked=True)

        for band_position, index_name in enumerate(indices):
            values = np.asarray(data[band_position].filled(np.nan), dtype=np.float64)
            valid_values = values[np.isfinite(values)]
            n_valid_pixels = int(valid_values.size)
            if n_valid_pixels > site_area_pixels:
                raise ValueError(f"{path} has more valid pixels than its footprint")
            pct_valid_pixels = (
                100.0 * n_valid_pixels / site_area_pixels if site_area_pixels else np.nan
            )
            valid_area_ha = n_valid_pixels * pixel_area_m2 / 10_000.0
            valid_area_km2 = n_valid_pixels * pixel_area_m2 / 1_000_000.0

            if n_valid_pixels:
                p25, p50, p75 = np.percentile(valid_values, [25.0, 50.0, 75.0])
                mean_value = float(np.mean(valid_values))
                std_value = float(np.std(valid_values, ddof=1)) if n_valid_pixels > 1 else np.nan
                p25, p50, p75 = float(p25), float(p50), float(p75)
            else:
                mean_value = std_value = p25 = p50 = p75 = np.nan

            records.append({
                "site": site,
                "year": year,
                "month": month,
                "window": window,
                "index": index_name,
                "n_valid_pixels": n_valid_pixels,
                "site_area_pixels": site_area_pixels,
                "pct_valid_pixels": pct_valid_pixels,
                "mean": mean_value,
                "std": std_value,
                "p25": p25,
                "p50": p50,
                "p75": p75,
                "total_area_ha": total_area_ha,
                "total_area_km2": total_area_km2,
                "valid_area_ha": valid_area_ha,
                "valid_area_km2": valid_area_km2,
                "n_scenes": int(qa_info["n_scenes"]),
                "mean_cloud": float(qa_info["mean_cloud"]),
            })

    return pd.DataFrame(records)

## 4 · Compute and Export Statistics to CSV

In [5]:
# Compute statistics across all composite rasters
stats_df = extract_imagery_statistics(
    tif_paths=tif_files,
    site_area_pixels_by_site=site_area_pixels_by_site,
    site_pixel_area=site_pixel_area,
    qa_manifest_path=QA_MANIFEST_PATH,
    indices=TARGET_INDICES,
)

# Export the complete statistics table
OUTPUT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
stats_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Exported {len(stats_df)} records to {OUTPUT_CSV_PATH}")

Exported 1176 records to /home/tshihadah/code/wadi-gaza-research/outputs/composites/imagery_stats.csv


## 5 · Data Preview and Quality Checks

In [6]:
# Preview structure and first records
print(f"Summary: {len(stats_df)} rows across {stats_df['site'].nunique()} sites, "
      f"{stats_df['window'].nunique()} composite windows, and {stats_df['index'].nunique()} indices.")

stats_df.head(12)

Summary: 1176 rows across 7 sites, 42 composite windows, and 4 indices.


,site,year,month,window,index,n_valid_pixels,site_area_pixels,pct_valid_pixels,mean,std,p25,p50,p75,total_area_ha,total_area_km2,valid_area_ha,valid_area_km2,n_scenes,mean_cloud
0,AlAhrashProtectorate,2019,01,2019-01,NDVI,33848,33848,100.0,0.196705,0.130599,0.090658,0.144806,0.274628,338.0877,3.380877,338.0877,3.380877,5,5.74
1,AlAhrashProtectorate,2019,01,2019-01,GNDVI,33848,33848,100.0,0.310632,0.113981,0.217765,0.265460,0.381605,338.0877,3.380877,338.0877,3.380877,5,5.74
2,AlAhrashProtectorate,2019,01,2019-01,SAVI,33848,33848,100.0,0.163541,0.085030,0.090414,0.136561,0.223468,338.0877,3.380877,338.0877,3.380877,5,5.74
3,AlAhrashProtectorate,2019,01,2019-01,NDRE,33848,33848,100.0,0.106442,0.089840,0.032901,0.074082,0.163031,338.0877,3.380877,338.0877,3.380877,5,5.74
4,AlAhrashProtectorate,2019,02,2019-02,NDVI,33848,33848,100.0,0.189456,0.117869,0.091578,0.146838,0.264056,338.0877,3.380877,338.0877,3.380877,3,3.07
5,AlAhrashProtectorate,2019,02,2019-02,GNDVI,33848,33848,100.0,0.306154,0.107307,0.218009,0.266184,0.374706,338.0877,3.380877,338.0877,3.380877,3,3.07
6,AlAhrashProtectorate,2019,02,2019-02,SAVI,33848,33848,100.0,0.159484,0.076669,0.091803,0.139102,0.216827,338.0877,3.380877,338.0877,3.380877,3,3.07
7,AlAhrashProtectorate,2019,02,2019-02,NDRE,33848,33848,100.0,0.101151,0.082604,0.032130,0.072662,0.154760,338.0877,3.380877,338.0877,3.380877,3,3.07
8,AlAhrashProtectorate,2019,03,2019-03,NDVI,33848,33848,100.0,0.182430,0.100499,0.096591,0.151977,0.248692,338.0877,3.380877,338.0877,3.380877,1,2.20
9,AlAhrashProtectorate,2019,03,2019-03,GNDVI,33848,33848,100.0,0.313302,0.087658,0.238923,0.286185,0.374218,338.0877,3.380877,338.0877,3.380877,1,2.20


### Site-level completeness and cloud masking summary

In [7]:
# Check valid area coverage summary by site (using NDVI as representative index)
site_summary = stats_df[stats_df["index"] == "NDVI"].groupby("site").agg(
    n_composites=("window", "count"),
    total_area_ha=("total_area_ha", "first"),
    min_pct_valid=("pct_valid_pixels", "min"),
    mean_pct_valid=("pct_valid_pixels", "mean"),
    max_pct_valid=("pct_valid_pixels", "max"),
    mean_scenes=("n_scenes", "mean"),
).round(2)

site_summary

,n_composites,total_area_ha,min_pct_valid,mean_pct_valid,max_pct_valid,mean_scenes
site,,,,,,
AlAhrashProtectorate,42,338.09,62.36,99.10,100.0,3.69
BesorStream,42,113.03,99.85,99.99,100.0,3.69
HaBesorNationalPark,42,918.64,99.93,100.00,100.0,3.69
HolotNizzanim,42,1198.48,98.81,99.94,100.0,3.48
HolotPalmahim,42,620.89,3.14,97.69,100.0,3.45
NahalRubin,42,78.56,97.80,99.73,100.0,3.45
WadiGaza,42,284.33,88.50,99.71,100.0,3.71
